# Train and evaluate the drowsiness project's eye-state model

This notebook contains the complete training workflow: inspect labeled eye images, split the data,
extract HOG features, tune a linear SVM on validation data, evaluate once on held-out test data,
and export the model and results for `streamlit_app.py`. Training runs on a CPU.

**What the existing model does:** `shape_predictor_68_face_landmarks.dat` is a pretrained dlib
landmark predictor, not a training dataset. It remains pretrained. We train an **open/closed-eye
classifier** from scratch here; the app combines its predictions with a sustained-closure timer.
Eye-state test scores do not measure driver fatigue detection accuracy.

**Run All:** select a Python 3.11/3.12 Jupyter kernel and run the cells from the top.
Supply real labeled **single-eye crops** to train and produce evaluation results. Full-face Alert/Drowsy images need a separate labeling
and eye-cropping preparation step; do not put those images directly into these class folders.
No training dataset is included in this repository. Without data, Run All completes with a
clear pending status and skips the training stages.


## 0. Check the notebook environment

Use a **Python 3.11 or 3.12** kernel. The setup uses the active kernel's Python and keeps
**NumPy 1.26.4 + OpenCV 4.10.0.84** together, matching the existing Anaconda scientific libraries.
It installs packages only when they are missing or the NumPy/OpenCV versions differ.

If the kernel still holds old libraries after installation, this cell displays **Restart needed**
as normal output. Choose **Kernel → Restart Kernel**, then **Run All**. Until then the remaining
cells skip work safely. A kernel restart is necessary to clear old binary modules from memory.

With compatible packages installed and a fresh kernel, setup runs without pip or a restart.
The unrelated gensim/FuzzyTM and old Streamlit/protobuf conflicts do not affect notebook training.
Install the frontend's requirements in its project environment when running Streamlit.


In [1]:
import importlib.metadata as package_metadata
import importlib
import subprocess
import sys

ENVIRONMENT_READY = False
CONFIG_READY = DATASET_READY = SPLIT_READY = MODEL_READY = EVALUATION_READY = EXPORT_READY = False
RUN_STATUS = "Environment check pending"
print("Notebook Python:", sys.executable)

def installed_version(distribution):
    try:
        return package_metadata.version(distribution)
    except package_metadata.PackageNotFoundError:
        return None

def prepare_environment():
    if sys.version_info[:2] not in ((3, 11), (3, 12)):
        return False, "Select a Python 3.11 or 3.12 kernel, then Run All."

    compatible_versions = {"numpy": "1.26.4", "opencv-python": "4.10.0.84"}
    training_packages = {
        "pandas": "pandas==2.2.3",
        "matplotlib": "matplotlib==3.8.4",
        "scipy": "scipy==1.11.4",
        "scikit-learn": "scikit-learn==1.5.2",
        "ipython": "ipython>=8,<10",
    }
    needs_repair = any(installed_version(name) != expected
                       for name, expected in compatible_versions.items())
    missing = [spec for name, spec in training_packages.items()
               if installed_version(name) is None]
    print("Installed NumPy:", installed_version("numpy"))
    print("Installed OpenCV:", installed_version("opencv-python"))

    if needs_repair or missing:
        specs = [f"{name}=={version}" for name, version in compatible_versions.items()]
        specs.extend(missing)
        print("Installing:", ", ".join(specs))
        # Remember whether binary modules were loaded BEFORE changing their files.
        loaded = any(name in sys.modules for name in
                     ("numpy", "cv2", "pandas", "matplotlib", "scipy", "sklearn"))
        result = subprocess.run([sys.executable, "-m", "pip", "install",
                                 "--only-binary=:all:", *specs],
                                capture_output=True, text=True)
        if result.stdout.strip():
            print(result.stdout)
        if result.stderr.strip():
            print("Package installer notes:\n" + result.stderr)
        if result.returncode != 0:
            return False, "Package installation did not finish. Resolve the installer message above, then rerun setup."
        importlib.invalidate_caches()
        if loaded:
            globals()["_NOTEBOOK_DEPENDENCIES_CHANGED"] = True

    loaded_numpy = sys.modules.get("numpy")
    if (globals().get("_NOTEBOOK_DEPENDENCIES_CHANGED", False)
            or (loaded_numpy is not None and loaded_numpy.__version__ != "1.26.4")):
        return False, "Restart needed: choose Kernel → Restart Kernel, then Run All. The packages are installed."
    return True, "Environment ready. Compatible training dependencies are installed."

ENVIRONMENT_READY, RUN_STATUS = prepare_environment()
print(RUN_STATUS)


Notebook Python: /home/lokesh-sharma/anaconda3/bin/python
Installed NumPy: 1.26.4
Installed OpenCV: 4.10.0.84
Environment ready. Compatible training dependencies are installed.


## 1. Dataset and configuration

Set `DATA_DIR` below. Folder names are case insensitive (`Open` / `Closed` also work).
Either supply existing splits:
```
data/eyes/
  train/open/       train/closed/
  validation/open/  validation/closed/
  test/open/        test/closed/
```
Or supply `data/eyes/open/` and `data/eyes/closed/` and let the notebook make a 60/20/20 split.
For subject-independent splitting, use `open/person_01/*.jpg` and `closed/person_01/*.jpg`,
with the same person/session identifier under both classes. Use at least five independent groups.
All images from a group stay in one split. Predefined splits are checked for group overlap too.

For flat folders, image-level splitting cannot rule out the same person or neighboring video
frames appearing in different sets. Prefer subject/session groups or independently collected splits.
Only use datasets you are permitted to train on. Collect crops matching `detection.crop_eye`
(padded eye bounding boxes) to reduce the difference between training and camera images.


In [2]:
from pathlib import Path
import os

CONFIG_READY = DATASET_READY = SPLIT_READY = MODEL_READY = EVALUATION_READY = EXPORT_READY = False
# If Jupyter starts elsewhere, set this to your Drowsiness-Detection folder.
PROJECT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (PROJECT_DIR, *PROJECT_DIR.parents)
                     if (path / 'eye_model.py').is_file()), None)
if not globals().get('ENVIRONMENT_READY', False):
    print('Configuration pending: run setup and follow its status message.')
elif PROJECT_ROOT is None:
    RUN_STATUS = 'Set PROJECT_DIR to the project folder containing eye_model.py, then rerun this cell.'
    print(RUN_STATUS)
else:
    from pathlib import Path
    import os
    import sys
    import json
    import hashlib
    import warnings
    from datetime import datetime, timezone

    import cv2
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    from sklearn.model_selection import train_test_split, GroupShuffleSplit
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import LinearSVC
    from sklearn.dummy import DummyClassifier
    from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                                 classification_report, ConfusionMatrixDisplay,
                                 RocCurveDisplay, roc_auc_score)

    ROOT = PROJECT_ROOT
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    from eye_model import FEATURE_VERSION, extract_features, prepare_eye, EyeClassifier

    # Edit this path to your labeled single-eye image dataset.
    DATA_DIR = Path(os.environ.get('DROWSINESS_DATA_DIR', ROOT / 'data' / 'eyes')).expanduser().resolve()
    OUTPUT_DIR = Path(os.environ.get('DROWSINESS_OUTPUT_DIR', ROOT)).resolve()
    MODEL_DIR, REPORT_DIR = OUTPUT_DIR / 'models', OUTPUT_DIR / 'reports'
    SEED = 42
    CLASS_NAMES = ['open', 'closed']  # open=0, closed=1; never change label ordering.
    EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
    np.random.seed(SEED)
    print('Dataset:', DATA_DIR)
    print('OpenCV:', cv2.__version__)
    CONFIG_READY = True


Dataset: /home/lokesh-sharma/Documents/Projects/Drowsiness-Detection/data/eyes
OpenCV: 4.10.0


## 2. Load, validate, and deduplicate images

Unreadable images and exact decoded-image duplicates within a partition are reported and skipped.
Conflicting labels and duplicates crossing predefined splits stop training. No sample is silently
relabelled. Subject identifiers are inferred only when images have a subject subfolder.

If the dataset is missing or fails validation, a **Training pending** message explains why.
The later cells skip training, evaluation, and export; they do not report invented scores.
After adding/fixing the data, Run All again. Unexpected programming or runtime faults remain
visible so they can be investigated.


In [3]:
DATASET_READY = SPLIT_READY = MODEL_READY = EVALUATION_READY = EXPORT_READY = False
if not globals().get('CONFIG_READY', False):
    print('Dataset check pending: configuration is not ready.')
else:
    try:
        if not DATA_DIR.is_dir():
            raise FileNotFoundError(
                f'No labeled dataset at {DATA_DIR}. The .dat landmark predictor is not training data. '
                'Add eye images under open/ and closed/, or update DATA_DIR above.')

        def class_folders(folder):
            children = {p.name.lower(): p for p in folder.iterdir() if p.is_dir()}
            missing = set(CLASS_NAMES) - set(children)
            if missing:
                raise ValueError(f'{folder} is missing class folders: {sorted(missing)}')
            extra = set(children) - set(CLASS_NAMES)
            if extra:
                print(f'Data note: ignoring other folders in {folder}: {sorted(extra)}')
            return [children[name] for name in CLASS_NAMES]

        children = {p.name.lower(): p for p in DATA_DIR.iterdir() if p.is_dir()}
        provided_splits = any(name in children for name in ('train', 'test', 'validation', 'val', 'valid'))
        if provided_splits:
            validation_name = next((n for n in ('validation', 'val', 'valid') if n in children), None)
            if 'train' not in children or 'test' not in children or validation_name is None:
                raise ValueError('Provide train/, validation/ (or val/), and test/, each with open/ and closed/.')
            partitions = {'train': children['train'], 'validation': children[validation_name], 'test': children['test']}
        else:
            partitions = {'unsplit': DATA_DIR}

        records, seen, skipped = [], {}, []
        for partition, folder in partitions.items():
            for label, class_folder in enumerate(class_folders(folder)):
                for path in sorted(class_folder.rglob('*')):
                    if not path.is_file() or path.suffix.lower() not in EXTENSIONS:
                        continue
                    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
                    if image is None:
                        skipped.append((str(path), 'unreadable'))
                        continue
                    digest = hashlib.sha256(str(image.shape).encode() + image.tobytes()).hexdigest()
                    if digest in seen:
                        previous = seen[digest]
                        if previous['label'] != label:
                            raise ValueError(f'Identical image has conflicting labels: {path} and {previous["path"]}')
                        if previous['partition'] != partition:
                            raise ValueError(f'Duplicate crosses predefined splits: {path} and {previous["path"]}')
                        skipped.append((str(path), 'duplicate'))
                        continue
                    relative = path.relative_to(class_folder)
                    group = relative.parts[0] if len(relative.parts) > 1 else None
                    record = {'path': str(path), 'label': label, 'class_name': CLASS_NAMES[label],
                              'group': group, 'partition': partition, 'sha256': digest}
                    seen[digest] = record
                    records.append(record)

        dataset = pd.DataFrame(records)
        if dataset.empty:
            raise ValueError('No readable labeled eye images found.')
        if dataset['label'].value_counts().reindex([0, 1], fill_value=0).min() < 5:
            raise ValueError('Need at least five distinct readable images per class; use substantially more for real training.')
        if dataset['group'].notna().any() and dataset['group'].isna().any():
            raise ValueError('Mixed flat and subject folders. Put every image in a subject/session folder, or use flat folders throughout.')
    except (FileNotFoundError, ValueError) as exc:
        RUN_STATUS = f'Training pending: {exc}'
        print(RUN_STATUS)
        print('No model or evaluation metrics will be generated in this run.')
    else:
        DATASET_READY = True
        print(f'Loaded {len(dataset)} images; skipped {len(skipped)} unreadable/duplicate files.')
        if skipped:
            display(pd.DataFrame(skipped, columns=['path', 'reason']).head(20))
        display(pd.crosstab(dataset['partition'], dataset['class_name']))
        fig, axes = plt.subplots(2, 5, figsize=(12, 5))
        for label in (0, 1):
            samples = dataset[dataset.label == label].sample(n=5, random_state=SEED)
            for ax, row in zip(axes[label], samples.itertuples()):
                ax.imshow(prepare_eye(cv2.imread(row.path)), cmap='gray')
                ax.set_title(CLASS_NAMES[label]); ax.axis('off')
        plt.suptitle('Preprocessed labeled eye crops'); plt.tight_layout(); plt.show()


Training pending: No labeled dataset at /home/lokesh-sharma/Documents/Projects/Drowsiness-Detection/data/eyes. The .dat landmark predictor is not training data. Add eye images under open/ and closed/, or update DATA_DIR above.
No model or evaluation metrics will be generated in this run.


## 3. Create training, validation, and untouched test sets

The scaler and SVM are fit on training data only during model selection. The validation set
chooses `C`. After selection, refit on training + validation and evaluate once on the test set.
Exact deduplication cannot detect all near-duplicates: review your source videos and identities.


In [4]:
SPLIT_READY = MODEL_READY = EVALUATION_READY = EXPORT_READY = False
if not globals().get('DATASET_READY', False):
    print('Split skipped: labeled training data is not ready.')
else:
    try:
        def group_holdout(frame, fraction, seed):
            splitter = GroupShuffleSplit(n_splits=100, test_size=fraction, random_state=seed)
            for a, b in splitter.split(frame, frame.label, frame.group):
                left, right = frame.iloc[a], frame.iloc[b]
                if left.label.nunique() == right.label.nunique() == 2:
                    return left.copy(), right.copy()
            raise ValueError('Cannot split groups with both classes present. Add more independently labeled subjects/sessions.')

        if provided_splits:
            train_df = dataset[dataset.partition == 'train'].copy()
            val_df = dataset[dataset.partition == 'validation'].copy()
            test_df = dataset[dataset.partition == 'test'].copy()
            split_protocol = 'provided splits'
        elif dataset.group.notna().all():
            if dataset.group.nunique() < 5:
                raise ValueError('Use at least five independent subject/session groups, or provide independent train/validation/test folders.')
            development_df, test_df = group_holdout(dataset, 0.2, SEED)
            train_df, val_df = group_holdout(development_df, 0.25, SEED + 1)
            split_protocol = 'subject/session group holdout (approximately 60/20/20)'
        else:
            print('Data note: image-level split may include the same subject/video across sets. Prefer subject/session folders.')
            development_df, test_df = train_test_split(dataset, test_size=0.2, stratify=dataset.label, random_state=SEED)
            train_df, val_df = train_test_split(development_df, test_size=0.25, stratify=development_df.label, random_state=SEED)
            split_protocol = 'stratified image holdout (60/20/20; identity overlap unknown)'

        for name, frame in [('train', train_df), ('validation', val_df), ('test', test_df)]:
            if frame.label.nunique() != 2:
                raise ValueError(f'{name} must contain both open and closed eyes.')
            print(name, len(frame), frame.class_name.value_counts().to_dict())
        if dataset.group.notna().all():
            for left, right in [(train_df, val_df), (train_df, test_df), (val_df, test_df)]:
                overlap = set(left.group) & set(right.group)
                if overlap:
                    raise ValueError(f'Subject/session groups cross splits: {sorted(overlap)[:10]}')
        print('Split protocol:', split_protocol)
    except ValueError as exc:
        RUN_STATUS = f'Training pending: {exc}'
        print(RUN_STATUS)
    else:
        SPLIT_READY = True


Split skipped: labeled training data is not ready.


## 4. Extract features and train the complete eye classifier

HOG summarizes local edges in each equalized 64×32 grayscale eye crop. A StandardScaler and
class-balanced linear SVM learn open/closed boundaries. The exact same feature function is used
in Streamlit. Validation balanced accuracy selects the regularization strength; test data is not
used for model selection. SVM decision margins are not confidence probabilities.


In [5]:
MODEL_READY = EVALUATION_READY = EXPORT_READY = False
if not globals().get('SPLIT_READY', False):
    print('Training skipped: the preceding stage is not ready.')
else:
    def features_for(frame):
        return np.stack([extract_features(cv2.imread(path)) for path in frame.path])

    X_train, X_val = features_for(train_df), features_for(val_df)
    y_train, y_val = train_df.label.to_numpy(), val_df.label.to_numpy()
    trials = []
    for c in (0.001, 0.01, 0.1, 1.0, 10.0):
        candidate = make_pipeline(StandardScaler(), LinearSVC(
            C=c, class_weight='balanced', dual=False, max_iter=20000, random_state=SEED))
        candidate.fit(X_train, y_train)
        trials.append({'C': c,
                       'train_balanced_accuracy': balanced_accuracy_score(y_train, candidate.predict(X_train)),
                       'validation_balanced_accuracy': balanced_accuracy_score(y_val, candidate.predict(X_val))})
    trials_df = pd.DataFrame(trials)
    display(trials_df)
    best_c = float(trials_df.loc[trials_df.validation_balanced_accuracy.idxmax(), 'C'])
    ax = trials_df.plot(x='C', y=['train_balanced_accuracy', 'validation_balanced_accuracy'],
                        logx=True, marker='o', ylim=(0, 1.05), title='Regularization selection (validation only)')
    ax.set_ylabel('Balanced accuracy'); plt.show()

    model = make_pipeline(StandardScaler(), LinearSVC(
        C=best_c, class_weight='balanced', dual=False, max_iter=20000, random_state=SEED))
    X_development = np.concatenate([X_train, X_val])
    y_development = np.concatenate([y_train, y_val])
    model.fit(X_development, y_development)
    print(f'Final model fitted on {len(y_development)} images with C={best_c}.')
    MODEL_READY = True


Training skipped: the preceding stage is not ready.


## 5. Evaluate once and output results

Report accuracy, balanced accuracy, closed-eye F1, ROC AUC, per-class precision/recall, a confusion
matrix, ROC curve, and example test predictions. A majority-class baseline provides context.
These are held-out **eye-state** scores, not a validated measure of drowsiness or driving safety.


In [6]:
EVALUATION_READY = EXPORT_READY = False
if not globals().get('MODEL_READY', False):
    print('Evaluation skipped: the preceding stage is not ready.')
else:
    X_test = features_for(test_df)
    y_test = test_df.label.to_numpy()
    y_pred = model.predict(X_test)
    margins = model.decision_function(X_test)
    baseline = DummyClassifier(strategy='most_frequent').fit(X_development, y_development)
    report = classification_report(y_test, y_pred, labels=[0, 1], target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    metrics = {
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'task': 'single-eye open/closed classification', 'feature_version': FEATURE_VERSION,
        'split_protocol': split_protocol, 'seed': SEED, 'selected_C': best_c,
        'counts': {'train': len(train_df), 'validation': len(val_df), 'test': len(test_df)},
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_test, y_pred)),
        'closed_f1': float(f1_score(y_test, y_pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_test, margins)),
        'majority_baseline_accuracy': float(accuracy_score(y_test, baseline.predict(X_test))),
        'majority_baseline_balanced_accuracy': float(balanced_accuracy_score(y_test, baseline.predict(X_test))),
        'classification_report': report,
    }
    display(pd.DataFrame(report).T)
    print(json.dumps({k: v for k, v in metrics.items() if k != 'classification_report'}, indent=2))
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=[0, 1],
                                            display_labels=CLASS_NAMES, ax=axes[0], colorbar=False)
    RocCurveDisplay.from_predictions(y_test, margins, ax=axes[1], name='closed eye')
    axes[0].set_title('Held-out confusion matrix')
    plt.tight_layout(); fig.savefig(REPORT_DIR / 'confusion_matrix.png', dpi=150); plt.show()

    predictions = test_df.reset_index(drop=True).copy()
    predictions['predicted_label'] = y_pred
    predictions['predicted_class'] = [CLASS_NAMES[int(p)] for p in y_pred]
    predictions['closed_margin'] = margins
    predictions.to_csv(REPORT_DIR / 'test_predictions.csv', index=False)
    fig, axes = plt.subplots(3, 4, figsize=(12, 8))
    examples = predictions.sample(n=min(12, len(predictions)), random_state=SEED)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, row in zip(axes.ravel(), examples.itertuples()):
        ax.imshow(prepare_eye(cv2.imread(row.path)), cmap='gray')
        ax.set_title(f'True: {row.class_name} | Pred: {row.predicted_class}',
                     color='green' if row.label == row.predicted_label else 'red', fontsize=9)
    plt.tight_layout(); fig.savefig(REPORT_DIR / 'example_predictions.png', dpi=150); plt.show()
    EVALUATION_READY = True


Evaluation skipped: the preceding stage is not ready.


## 6. Export the trained model and verify inference matches

The app loads numeric arrays from an NPZ file without pickle. Verify that exported inference
matches the fitted scikit-learn pipeline before writing the final report. The model stays fitted
on training + validation, preserving the held-out meaning of the reported test score.


In [7]:
EXPORT_READY = False
if not globals().get('EVALUATION_READY', False):
    print('Export skipped: the preceding stage is not ready.')
else:
    scaler, svm = model.named_steps['standardscaler'], model.named_steps['linearsvc']
    assert np.array_equal(svm.classes_, [0, 1])
    model_path = MODEL_DIR / 'eye_state_model.npz'
    np.savez_compressed(model_path, feature_version=np.array(FEATURE_VERSION),
                        mean=scaler.mean_, scale=scaler.scale_,
                        coef=svm.coef_, intercept=svm.intercept_[0])
    exported = EyeClassifier(model_path)
    check_images = [cv2.imread(path) for path in test_df.path.iloc[:20]]
    np.testing.assert_allclose(exported.decision_function(check_images),
                               margins[:len(check_images)], rtol=1e-5, atol=1e-6)
    np.testing.assert_array_equal(exported.predict(check_images), y_pred[:len(check_images)])
    metrics['model_sha256'] = hashlib.sha256(model_path.read_bytes()).hexdigest()
    (REPORT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
    trials_df.to_csv(REPORT_DIR / 'validation_trials.csv', index=False)
    manifest = pd.concat([train_df.assign(split='train'), val_df.assign(split='validation'),
                          test_df.assign(split='test')], ignore_index=True)
    manifest.to_csv(REPORT_DIR / 'split_manifest.csv', index=False)
    print('Verified exported predictions match scikit-learn.')
    print('Model:', model_path)
    print('Results:', REPORT_DIR)
    EXPORT_READY = True


Export skipped: the preceding stage is not ready.


In [8]:
if globals().get('EXPORT_READY', False):
    RUN_STATUS = 'Training, evaluation, and model export completed.'
print('Run summary:', RUN_STATUS)
if not globals().get('EXPORT_READY', False):
    print('No new model or test metrics were produced. Complete the step described above, then Run All.')


Run summary: Training pending: No labeled dataset at /home/lokesh-sharma/Documents/Projects/Drowsiness-Detection/data/eyes. The .dat landmark predictor is not training data. Add eye images under open/ and closed/, or update DATA_DIR above.
No new model or test metrics were produced. Complete the step described above, then Run All.


## 7. Run the frontend

From the project directory, in the same Python environment:
```bash
streamlit run streamlit_app.py
```
Choose **Trained eye classifier**. The app also offers the original **EAR threshold** method,
which works with the existing landmark predictor before training. Test a face photo, a video,
or browser webcam. A single image cannot establish sustained eye closure. In live/video mode,
a drowsiness alert requires both eyes to remain classified closed for the configured duration.
No face, an observation gap, or a large face-box change resets the closure timer.

Review failures on unseen people, glasses, low light, head poses, and partially visible eyes.
Misaligned training crops and neighboring frames across splits can produce misleading scores.
Improve the labeled data and validation design before interpreting high accuracy as generalization.

References: [LinearSVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html),
[GroupShuffleSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html),
[Streamlit camera input](https://docs.streamlit.io/develop/api-reference/widgets/st.camera_input),
[streamlit-webrtc](https://github.com/whitphx/streamlit-webrtc).
